# 01 - EDA: Looking at the Raw Data

**EDA = Exploratory Data Analysis** - before any modeling, you *look* at the data to confirm it loaded correctly and to build intuition for what a flare looks like.

**How to use this notebook:** run each grey code cell in order with **Shift+Enter**. Cells build on the ones above, so go top to bottom. A `.ipynb` notebook runs code in small chunks and shows charts right under each cell.

**What you'll do:** open one raw SoLEXS file to see its structure, plot a GOES day and a SoLEXS day (their "light curves"), and look at the flare catalog.

**Before running:** you need GOES `.nc` files in `data/raw/goes/`, SoLEXS folders in `data/raw/aditya_l1/solexs/`, and a built catalog (`python scripts/process_all.py --rebuild`).

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import yaml
from pathlib import Path
from astropy.io import fits

DATA = Path('../data')
with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)
print('config loaded; n_classes =', cfg['forecasting'].get('n_classes', 0))

## 1. Peek inside a SoLEXS file

Space data comes in **FITS** files (a standard astronomy format). One FITS file holds several sections called **HDUs** (Header/Data Units) - think of them like sheets in a spreadsheet. SoLEXS light curves are gzip-compressed FITS (`.lc.gz`) inside per-day folders; the `SDD2` folder is the primary detector.

This cell opens one file and prints its structure so you can see the table columns (like `TIME` and `COUNTS`). You normally never do this by hand - the readers in `pipeline/readers.py` do it for you - but it's worth seeing once.

In [ ]:
lc_files = sorted(DATA.glob('raw/aditya_l1/solexs/**/SDD2/*.lc.gz'))
print(f'Found {len(lc_files)} SDD2 light-curve files')
if lc_files:
    with fits.open(lc_files[0]) as hdul:
        hdul.info()
        print('\nRATE columns:', hdul['RATE'].data.dtype.names)
else:
    print('No SoLEXS .lc.gz under data/raw/aditya_l1/solexs/ — check the download.')

## 2. Plot a GOES day (our training "textbook")

A **light curve** is just a graph of X-ray brightness (flux) over time. Flat = quiet Sun; a sharp spike = a flare.

GOES is a US satellite with decades of clean data in real physical units (**W/m^2**), which is why we train the model on it first. It has two soft-X-ray channels: **1-8 A** (`flux_long`, the official flare-size channel) and **0.5-4 A** (`flux_short`). The dashed orange line is the **background** (the quiet baseline the code estimates) - flares are bumps that rise clearly above it. The y-axis is logarithmic because flares can be thousands of times brighter than the quiet Sun.

In [ ]:
from pipeline.readers import read_goes_nc
from pipeline.preprocessor import preprocess_goes

goes_files = sorted(DATA.glob('raw/goes/sci_xrsf-l2-avg1m_g16_d*.nc'))
print(f'{len(goes_files)} GOES days available')
if goes_files:
    proc = preprocess_goes(read_goes_nc(goes_files[len(goes_files)//2]), cfg)  # a mid-range day
    fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
    axes[0].semilogy(proc.index, proc['flux_long'], lw=0.6, label='1-8 A')
    axes[0].semilogy(proc.index, proc['bg_long'], '--', color='orange', lw=1, label='background')
    axes[0].set_ylabel('W/m^2'); axes[0].set_title('GOES XRS soft X-ray'); axes[0].legend()
    axes[1].semilogy(proc.index, proc['flux_short'], lw=0.6, color='green', label='0.5-4 A')
    axes[1].set_ylabel('W/m^2'); axes[1].legend()
    plt.tight_layout(); plt.show()

## 3. Plot a real SoLEXS day (the actual Aditya-L1 data)

Same idea, but now the real instrument you're judged on. The units here are **counts/s** (raw detector clicks), *not* W/m^2 - converting between them needs a calibration step we do later. So the *shape* is meaningful (spikes = flares), but the absolute class (C vs M vs X) depends on that calibration.

In [ ]:
sxr_files = sorted(DATA.glob('processed/*_solexs.parquet'))
print(f'{len(sxr_files)} processed SoLEXS days')
if sxr_files:
    sx = pd.read_parquet(sxr_files[0])
    print(sx[['flux_long']].describe())
    fig, ax = plt.subplots(figsize=(13, 3))
    ax.plot(sx.index, sx['flux_long'], lw=0.5, label='flux_long')
    if 'bg_long' in sx:
        ax.plot(sx.index, sx['bg_long'], '--', color='orange', lw=1, label='background')
    ax.set_yscale('log'); ax.set_ylabel('counts/s')
    ax.set_title(f'SoLEXS SDD2 — {sxr_files[0].name[:8]}'); ax.legend()
    plt.tight_layout(); plt.show()

## 4. The flare catalog

The **catalog** is the master list of every flare the detector found, with its time and strength **class** (A/B/C/M/X, weakest to strongest - each step is 10x stronger). It was produced by `scripts/process_all.py`. The bar chart shows how many of each class; you'll usually see many small C flares and few big X flares, which is normal (strong flares are rare).

In [ ]:
from pipeline.catalog import load_master_catalog, catalog_summary
cat = load_master_catalog('../data/processed/master_catalog.parquet')
print(catalog_summary(cat))
if not cat.empty:
    counts = cat['flare_class'].str[0].value_counts().reindex(list('ABCMX')).fillna(0)
    counts.plot(kind='bar', color='steelblue', figsize=(6, 3), title='Flares by class')
    plt.ylabel('count'); plt.tight_layout(); plt.show()
cat.head()